# AI Penetration Across Scientific Fields
Exploratory analysis of how AI/ML methods spread across disciplines (2000-2025).

**Research Question**: How has AI adoption varied across scientific fields, and what patterns emerge in the diffusion timeline?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("../data/processed")

In [ ]:
df = pd.read_parquet(DATA_DIR / "ai_adoption_by_field.parquet")
print(f"Rows: {len(df)}, Fields: {df['field_name'].nunique()}")
df.head(10)

## 1. AI Adoption Timeline by Field


In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
for field_name, group in df.groupby("field_name"):
    ax.plot(group["year"], group["ai_fraction"] * 100, label=field_name, linewidth=2)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("AI-related Works (%)", fontsize=12)
ax.set_title("AI Adoption Rate by Scientific Field (2000-2025)", fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Absolute AI Works Growth


In [ ]:
pivot = df.pivot_table(
    index="year", columns="field_name", values="ai_count", fill_value=0
)
pivot.plot.area(figsize=(16, 10), alpha=0.7)
plt.title("Absolute Number of AI-related Works by Field", fontsize=14)
plt.ylabel("Number of AI Works")
plt.xlabel("Year")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Heatmap: AI Adoption Intensity


In [ ]:
heatmap_data = df.pivot_table(index="field_name", columns="year", values="ai_fraction")
fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(heatmap_data * 100, annot=False, cmap="YlOrRd", ax=ax, fmt=".1f")
ax.set_title("AI Adoption Rate (%) by Field and Year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Field")
plt.tight_layout()
plt.show()

## 4. Key Statistics


In [ ]:
latest = df[df["year"] == df["year"].max()]
latest_sorted = latest.sort_values("ai_fraction", ascending=False)
print("=== Latest Year AI Adoption Ranking ===")
for _, row in latest_sorted.iterrows():
    print(
        f"  {row['field_name']:50s} {row['ai_fraction'] * 100:6.2f}%  ({int(row['ai_count']):,} AI works / {int(row['total_count']):,} total)"
    )

## 5. Acceleration Analysis
Which fields showed the fastest AI adoption acceleration in the last 5 years?


In [ ]:
recent = df[df["year"] >= 2020].copy()
early = df[(df["year"] >= 2015) & (df["year"] < 2020)].copy()

recent_avg = recent.groupby("field_name")["ai_fraction"].mean()
early_avg = early.groupby("field_name")["ai_fraction"].mean()
acceleration = (recent_avg - early_avg).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
acceleration.plot.barh(ax=ax, color=sns.color_palette("husl", len(acceleration)))
ax.set_xlabel("Change in AI Fraction (2020-2025 vs 2015-2019)")
ax.set_title("AI Adoption Acceleration by Field")
plt.tight_layout()
plt.show()